# 00 - Data Setup

This notebook creates the demo database, schemas, and loads synthetic customer churn data.

**Prerequisites**: Snowflake account with permissions to create databases and schemas.

**Database**: `MLOPS_DEMO_DB`  
**Schemas**: RAW, FEATURES, PIPELINES, OUTPUT, MONITORING

In [ ]:
from snowflake.snowpark.context import get_active_session
session = get_active_session()
print(f"Connected as: {session.get_current_user()}")
print(f"Role: {session.get_current_role()}")

In [ ]:
# Create database and schemas
session.sql("CREATE DATABASE IF NOT EXISTS MLOPS_DEMO_DB").collect()
session.sql("CREATE SCHEMA IF NOT EXISTS MLOPS_DEMO_DB.RAW").collect()
session.sql("CREATE SCHEMA IF NOT EXISTS MLOPS_DEMO_DB.FEATURES").collect()
session.sql("CREATE SCHEMA IF NOT EXISTS MLOPS_DEMO_DB.PIPELINES").collect()
session.sql("CREATE SCHEMA IF NOT EXISTS MLOPS_DEMO_DB.OUTPUT").collect()
session.sql("CREATE SCHEMA IF NOT EXISTS MLOPS_DEMO_DB.MONITORING").collect()
print("Database and schemas created.")

In [ ]:
# Create the CUSTOMERS table with synthetic churn data (~1000 rows)
session.sql("""
CREATE OR REPLACE TABLE MLOPS_DEMO_DB.RAW.CUSTOMERS AS
SELECT
    ROW_NUMBER() OVER (ORDER BY SEQ4()) AS CUSTOMER_ID,
    UNIFORM(350, 850, RANDOM()) AS CREDIT_SCORE,
    CASE UNIFORM(1, 3, RANDOM())
        WHEN 1 THEN 'France'
        WHEN 2 THEN 'Germany'
        ELSE 'Spain'
    END AS GEOGRAPHY,
    CASE WHEN UNIFORM(0, 1, RANDOM()) = 0 THEN 'Male' ELSE 'Female' END AS GENDER,
    UNIFORM(18, 75, RANDOM()) AS AGE,
    UNIFORM(0, 10, RANDOM()) AS TENURE,
    ROUND(UNIFORM(0, 250000, RANDOM())::FLOAT, 2) AS BALANCE,
    UNIFORM(1, 4, RANDOM()) AS NUM_OF_PRODUCTS,
    UNIFORM(0, 1, RANDOM()) AS HAS_CR_CARD,
    UNIFORM(0, 1, RANDOM()) AS IS_ACTIVE_MEMBER,
    ROUND(UNIFORM(10000, 200000, RANDOM())::FLOAT, 2) AS ESTIMATED_SALARY,
    -- Target: ~20% churn rate, influenced by age and balance
    CASE
        WHEN UNIFORM(0, 100, RANDOM()) < 20 THEN 1
        ELSE 0
    END AS EXITED
FROM TABLE(GENERATOR(ROWCOUNT => 1000))
""").collect()
print("CUSTOMERS table created with 1000 rows.")

In [ ]:
# Explore the data
df = session.table("MLOPS_DEMO_DB.RAW.CUSTOMERS")
print(f"Total rows: {df.count()}")
print(f"\nSchema:")
for field in df.schema.fields:
    print(f"  {field.name}: {field.datatype}")

In [ ]:
# Class balance (churn rate)
session.sql("""
SELECT 
    EXITED,
    COUNT(*) AS COUNT,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 1) AS PERCENTAGE
FROM MLOPS_DEMO_DB.RAW.CUSTOMERS
GROUP BY EXITED
ORDER BY EXITED
""").show()

In [ ]:
# Summary statistics
session.sql("""
SELECT
    COUNT(*) AS total_customers,
    ROUND(AVG(CREDIT_SCORE), 1) AS avg_credit_score,
    ROUND(AVG(AGE), 1) AS avg_age,
    ROUND(AVG(TENURE), 1) AS avg_tenure,
    ROUND(AVG(BALANCE), 2) AS avg_balance,
    ROUND(AVG(ESTIMATED_SALARY), 2) AS avg_salary,
    ROUND(AVG(NUM_OF_PRODUCTS), 1) AS avg_products
FROM MLOPS_DEMO_DB.RAW.CUSTOMERS
""").show()

In [ ]:
# Distribution by geography
session.sql("""
SELECT 
    GEOGRAPHY,
    COUNT(*) AS TOTAL,
    SUM(EXITED) AS CHURNED,
    ROUND(SUM(EXITED) * 100.0 / COUNT(*), 1) AS CHURN_RATE_PCT
FROM MLOPS_DEMO_DB.RAW.CUSTOMERS
GROUP BY GEOGRAPHY
ORDER BY GEOGRAPHY
""").show()

## What to Show in SnowSight

Navigate to **Data > Databases > MLOPS_DEMO_DB > RAW > CUSTOMERS** to see:
- Table structure and column types
- Data preview
- Table details (row count, size)

---

**Next**: `01_feature_engineering.ipynb` - Create features using the Feature Store